<a href="https://colab.research.google.com/github/diazcas2/mIArmario/blob/main/MODULO4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montado")

In [ ]:
import os

base = "/content/drive/MyDrive"
print("📁 Carpetas en tu Drive que contienen 'ARMARIO':")
for carpeta in os.listdir(base):
    if "ARMARIO" in carpeta.upper():
        print(f"\n✅ '{carpeta}'")
        ruta = os.path.join(base, carpeta)
        for archivo in sorted(os.listdir(ruta)):
            print(f"   - {archivo}")

In [ ]:
import json

RUTA_JSON = "/content/drive/MyDrive/ARMARIO/armario.json"
RUTA_BASE = "/content/drive/MyDrive/ARMARIO"

with open(RUTA_JSON, "r") as f:
    armario = json.load(f)

for prenda in armario:
    nombre_archivo = prenda["imagen_path"].split("/")[-1]
    prenda["imagen_path"] = f"{RUTA_BASE}/{nombre_archivo}"

print("✅ Rutas corregidas:")
for p in armario:
    existe = "✅" if __import__('os').path.exists(p["imagen_path"]) else "❌"
    print(f"  {existe} {p['tipo']} — {p['imagen_path'].split('/')[-1]}")

In [ ]:
from PIL import Image
import io, os, requests
from IPython.display import display, Image as IPImage

def cargar_imagen(path):
    """Carga imagen desde archivo local o URL."""
    if path.startswith("http"):
        r = requests.get(path)
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    return Image.open(path).convert("RGB")

print("✅ Imports listos")

In [ ]:
def describir_outfit(foto_persona, lista_prendas, estilo_ocasion):
    img_persona = cargar_imagen(foto_persona)
    imagenes_prendas = []
    nombres_prendas = []

    for prenda in lista_prendas:
        if not os.path.exists(prenda["path"]):
            print(f"❌ Archivo no encontrado: {prenda['path']}")
            continue
        imagenes_prendas.append(cargar_imagen(prenda["path"]))
        nombres_prendas.append(f"{prenda['nombre']} ({prenda.get('categoria', 'prenda')})")

    if not imagenes_prendas:
        raise ValueError("No hay prendas válidas")

    lista_texto = "\n".join([f"- {n}" for n in nombres_prendas])

    prompt_vision = f"""Describe in English, in detail, how this person would look
wearing ALL of the following garments together as a complete outfit for: {estilo_ocasion}

Garments:
{lista_texto}

Include:
- Exact physical appearance of the person (age, build, hair, skin tone, face)
- Each garment with its color, texture, fit and style
- How all pieces combine as a full outfit
- Overall style and mood

Be very specific and visual. Maximum 250 words."""

    print("👁️  Analizando persona y prendas...")
    vision_model = genai.GenerativeModel("gemini-2.5-flash")
    contenido = [prompt_vision, img_persona] + imagenes_prendas
    response = vision_model.generate_content(contenido)

    descripcion = response.text
    print(f"\n📝 Descripción generada:\n{'-'*50}\n{descripcion}\n{'-'*50}")
    return descripcion


In [ ]:
from PIL import Image, ImageDraw, ImageOps
import os

def quitar_fondo_blanco(img, umbral=240):
    """Recorta el espacio vacío blanco alrededor de la prenda."""
    img_rgb = img.convert("RGB")
    bbox = img_rgb.getbbox()
    pixels = img_rgb.load()
    # Encontrar bbox real sin fondo blanco
    min_x, min_y = img.width, img.height
    max_x, max_y = 0, 0
    for y in range(img.height):
        for x in range(img.width):
            r, g, b = pixels[x, y]
            if not (r > umbral and g > umbral and b > umbral):
                min_x = min(min_x, x)
                min_y = min(min_y, y)
                max_x = max(max_x, x)
                max_y = max(max_y, y)
    if max_x > min_x and max_y > min_y:
        return img.crop((min_x, min_y, max_x + 1, max_y + 1))
    return img

def generar_outfit_vertical(lista_prendas, ocasion, descripcion=""):
    """
    Layout vertical: prendas apiladas de arriba a abajo en orden outfit.
    Upper body → lower body → shoes. Fondo blanco, estilo lookbook.
    """
    ANCHO         = 600
    ALTO_PRENDA   = 400
    PADDING       = 30
    COLOR_FONDO   = (255, 255, 255)
    COLOR_LINEA   = (235, 235, 235)
    COLOR_TEXTO   = (40, 40, 40)
    COLOR_GRIS    = (150, 150, 150)
    COLOR_ACENTO  = (30, 30, 30)

    n_prendas  = len(lista_prendas)
    alto_total = (ALTO_PRENDA + PADDING) * n_prendas + 160

    canvas = Image.new("RGB", (ANCHO, alto_total), COLOR_FONDO)
    draw   = ImageDraw.Draw(canvas)

    # ── Título ────────────────────────────────────────────────────────────────
    draw.text((ANCHO // 2, 38), "OUTFIT COMPLETO",
              fill=COLOR_ACENTO, anchor="mm")
    draw.text((ANCHO // 2, 62), ocasion.upper(),
              fill=COLOR_GRIS, anchor="mm")
    draw.line([(PADDING * 2, 80), (ANCHO - PADDING * 2, 80)],
              fill=COLOR_LINEA, width=1)

    # ── Prendas ───────────────────────────────────────────────────────────────
    y_actual = 100

    for i, prenda in enumerate(lista_prendas):
        y_celda = y_actual
        x_celda = PADDING

        # Separador entre prendas
        if i > 0:
            draw.line([(PADDING * 3, y_celda - PADDING // 2),
                       (ANCHO - PADDING * 3, y_celda - PADDING // 2)],
                      fill=COLOR_LINEA, width=1)

        # Imagen de la prenda
        try:
            img_prenda = Image.open(prenda["imagen_path"]).convert("RGB")
            img_prenda = quitar_fondo_blanco(img_prenda)

            max_w = ANCHO - PADDING * 2
            max_h = ALTO_PRENDA - 50
            ratio = min(max_w / img_prenda.width, max_h / img_prenda.height)
            pw    = int(img_prenda.width  * ratio)
            ph    = int(img_prenda.height * ratio)
            img_prenda = img_prenda.resize((pw, ph), Image.LANCZOS)

            px = (ANCHO - pw) // 2
            py = y_celda + (max_h - ph) // 2
            canvas.paste(img_prenda, (px, py))

        except Exception as e:
            draw.text((ANCHO // 2, y_celda + ALTO_PRENDA // 2),
                      f"[{prenda.get('tipo', 'prenda')}]",
                      fill=COLOR_GRIS, anchor="mm")
            print(f"⚠️ Error con {prenda.get('imagen_path')}: {e}")

        # Etiqueta debajo de la imagen
        etiqueta = f"{prenda.get('tipo','').upper()}  ·  {prenda.get('color','')}"
        draw.text((ANCHO // 2, y_celda + ALTO_PRENDA - 15),
                  etiqueta, fill=COLOR_GRIS, anchor="mm")

        y_actual += ALTO_PRENDA + PADDING

    # ── Pie: descripción de Gemini ────────────────────────────────────────────
    if descripcion:
        draw.line([(PADDING * 2, y_actual),
                   (ANCHO - PADDING * 2, y_actual)],
                  fill=COLOR_LINEA, width=1)
        # Partir descripción en líneas de ~70 chars
        palabras    = descripcion.split()
        lineas      = []
        linea_actual = ""
        for palabra in palabras:
            if len(linea_actual) + len(palabra) + 1 <= 70:
                linea_actual += (" " if linea_actual else "") + palabra
            else:
                lineas.append(linea_actual)
                linea_actual = palabra
        if linea_actual:
            lineas.append(linea_actual)

        y_texto = y_actual + 20
        for linea in lineas[:4]:  # Máximo 4 líneas
            draw.text((ANCHO // 2, y_texto), linea,
                      fill=COLOR_GRIS, anchor="mm")
            y_texto += 22

    # ── Guardar ───────────────────────────────────────────────────────────────
    ruta_salida = "/content/outfit_final.jpg"
    canvas.save(ruta_salida, quality=95)
    print(f"✅ Outfit guardado en {ruta_salida}")
    return ruta_salida

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from PIL import Image as PILImage
from IPython.display import display, Image as IPImage
import requests, io, os, json

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

def describir_outfit(lista_prendas, estilo_ocasion):
    nombres = [f"{p.get('tipo')} {p.get('color','')}" for p in lista_prendas]
    lista_texto = ", ".join(nombres)
    prompt = f"""En máximo 2 frases en español, describe si este outfit
es apropiado para: {estilo_ocasion}
Prendas: {lista_texto}"""
    model    = genai.GenerativeModel("gemini-2.5-flash")
    response = model.generate_content(prompt)
    return response.text.strip()

def modulo_4(prendas_seleccionadas, ocasion_detectada):
    """
    Entrada: lista de prendas del armario.json + ocasión del Módulo 1
    Salida:  imagen del outfit en layout vertical
    """

    descripcion = describir_outfit(prendas_seleccionadas, ocasion_detectada)
    print(f"📝 {descripcion}\n")

    ruta = generar_outfit_vertical(prendas_seleccionadas, ocasion_detectada, descripcion)

    print("\n✨ ¡Outfit generado!")
    display(IPImage(filename=ruta))
    return ruta

print("✅ Módulo 4 listo")

In [ ]:
prendas_outfit = [
    next(p for p in armario if p["id"] == "prenda_001_1"),  # chaqueta azul
    next(p for p in armario if p["id"] == "prenda_002_1"),  # pantalón negro
    next(p for p in armario if p["id"] == "prenda_008_1"),  # mocasines marrón
]

mi_ocasion = "cena de empresa, look smart casual"
print("✅ Prendas seleccionadas:")
for p in prendas_outfit:
    print(f"  - {p['tipo']} {p['color']} → {p['imagen_path'].split('/')[-1]}")

In [ ]:
resultado = modulo_4(prendas_outfit, mi_ocasion)